# Notebook 04 — Analyse Exploratoire Statistique (EDA)

**Projet :** Prediction du risque d'abandon scolaire  
**Equipe :** Hugo RAGUIN · Amine TALEB · Elliot FIORESE  

Ce notebook realise une **analyse statistique approfondie** :

1. Statistiques descriptives globales et par groupe
2. Analyse du desequilibre de classes
3. Correlations et relations de causalite
4. Identification des biais potentiels
5. Analyse bivariee des variables les plus discriminantes

In [ ]:
import os
import sys

PROJECT_ROOT = os.path.abspath('..')
os.chdir(PROJECT_ROOT)
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from utils_viz import set_custom_style

set_custom_style(theme='light')
%matplotlib inline

df = pd.read_csv('data/processed/tp1_student_risk_wrangled.csv')
df['dropout_risk'] = df['dropout_risk'].astype(bool)
df['scholarship'] = df['scholarship'].astype(bool)
df['internet_access'] = df['internet_access'].astype(bool)

RISK = df['dropout_risk']
df_risk = df[RISK == True]
df_norisk = df[RISK == False]

print(f'Dataset : {df.shape}  |  A risque : {RISK.sum()} ({RISK.mean()*100:.1f}%)')

## 1. Statistiques descriptives globales

In [ ]:
numeric_cols = [
    'age', 'study_hours_per_week', 'lms_sessions_week', 'attendance_rate',
    'assignment_delay_days', 'prior_average', 'continuous_assessment',
    'stress_index', 'commute_minutes', 'engagement_score',
    'grade_trend_gap', 'academic_pressure_index'
]
df[numeric_cols].describe().round(2)

In [ ]:
# Statistiques comparees : a risque vs. non a risque
compare_cols = [
    'attendance_rate', 'study_hours_per_week', 'prior_average',
    'continuous_assessment', 'assignment_delay_days', 'stress_index',
    'engagement_score', 'academic_pressure_index'
]
stats_comparison = pd.DataFrame({
    'Non a risque (moy)': df_norisk[compare_cols].mean(),
    'A risque (moy)': df_risk[compare_cols].mean(),
    'Non a risque (std)': df_norisk[compare_cols].std(),
    'A risque (std)': df_risk[compare_cols].std(),
}).round(2)
print('=== Comparaison des moyennes ===')
print(stats_comparison.to_string())

## 2. Analyse du desequilibre de classes

Le desequilibre de classes est un **biais structurel** important. Un modele qui predit toujours la classe majoritaire obtient une accuracy de ~91 % sans rien apprendre. Il faut donc corriger ce biais par :
- Un split **stratifie**
- L'usage de **`class_weight='balanced'`** dans les modeles
- Des metriques adaptees : **rappel, F1, ROC-AUC** (pas l'accuracy)

In [ ]:
n_pos = RISK.sum()
n_neg = (~RISK).sum()
ratio = n_neg / n_pos

print(f'Classe positive (a risque)    : {n_pos} ({n_pos/len(df)*100:.1f} %)')
print(f'Classe negative (non a risque) : {n_neg} ({n_neg/len(df)*100:.1f} %)')
print(f'Ratio majoritaire/minoritaire  : {ratio:.1f}:1')
print()
print('=> Desequilibre modere. Strategies adoptees :')
print('   - class_weight="balanced" dans Logistic Regression')
print('   - class_weight="balanced_subsample" dans Random Forest')
print('   - Split stratifie pour la validation croisee')

## 3. Correlations et analyse statistique approfondie

In [ ]:
# Correlation de chaque variable avec la cible
df_numeric = df[compare_cols].copy()
df_numeric['dropout_risk'] = RISK.astype(int)

corr_with_target = df_numeric.corr()['dropout_risk'].drop('dropout_risk').sort_values(key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
colors = ['#D93025' if v > 0 else '#188038' for v in corr_with_target.values]
bars = ax.barh(corr_with_target.index, corr_with_target.values, color=colors, edgecolor='none')
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xlabel('Correlation de Pearson avec dropout_risk')
ax.set_title('Correlation de chaque variable avec la cible')
ax.bar_label(bars, fmt='%.3f', padding=2, fontsize=8)
fig.tight_layout()
plt.show()

In [ ]:
# Test de Mann-Whitney U : les distributions sont-elles significativement differentes ?
print('=== Tests de Mann-Whitney U (p-value < 0.05 = difference significative) ===')
for col in compare_cols:
    group_risk = df_risk[col].dropna()
    group_norisk = df_norisk[col].dropna()
    stat, pval = stats.mannwhitneyu(group_risk, group_norisk, alternative='two-sided')
    sig = '***' if pval < 0.001 else '**' if pval < 0.01 else '*' if pval < 0.05 else 'ns'
    print(f'  {col:30s} : p = {pval:.4f}  {sig}')

## 4. Identification des biais potentiels

Un systeme d'alerte precoce peut amplifier des biais existants si les variables utilisees sont des proxys de caracteristiques sociales. Nous identifions ici les risques de biais.

In [ ]:
# Analyse des variables sociales vs. risque
social_vars = ['scholarship', 'internet_access', 'parental_education', 'program']

print('=== Taux de risque par variable sociale ===')
for var in social_vars:
    group_risk = df.groupby(var)['dropout_risk'].mean() * 100
    print(f'\n  {var} :')
    for cat, rate in group_risk.sort_values(ascending=False).items():
        n = df[df[var] == cat].shape[0]
        print(f'    {str(cat):20s} : {rate:.1f} %  (n={n})')

In [ ]:
# Heatmap : taux de risque par programme x education parentale
pivot = df.pivot_table(
    values='dropout_risk', index='program', columns='parental_education', aggfunc='mean'
) * 100

fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(pivot.round(1), annot=True, fmt='.1f', cmap='RdYlGn_r',
            ax=ax, linewidths=0.5, cbar_kws={'label': 'Taux de risque (%)'})
ax.set_title('Taux de risque (%) par Programme x Education parentale')
ax.set_xlabel('Education parentale')
ax.set_ylabel('Programme')
fig.tight_layout()
plt.show()

print('=> Les etudiants de Digital Design avec parents de niveau secondaire ont le risque le plus eleve.')
print('=> Biais potentiel : programme + contexte familial cumulent les desavantages.')

## 5. Analyse bivariee des variables les plus discriminantes

In [ ]:
# Pairplot des 4 variables les plus correlee avec la cible
top4 = corr_with_target.abs().head(4).index.tolist()
df_pair = df[top4 + ['dropout_risk']].copy()
df_pair['Statut'] = df_pair['dropout_risk'].map({True: 'A risque', False: 'Non a risque'})

g = sns.pairplot(
    df_pair.drop(columns='dropout_risk'),
    hue='Statut',
    palette={'Non a risque': '#188038', 'A risque': '#D93025'},
    plot_kws={'alpha': 0.4, 's': 15},
    diag_kind='kde'
)
g.figure.suptitle('Pairplot des 4 variables les plus discriminantes', y=1.02, fontsize=12)
plt.show()

In [ ]:
# Profil par programme : tableau complet
program_profile = (
    df.groupby('program')[[
        'attendance_rate', 'study_hours_per_week', 'prior_average',
        'continuous_assessment', 'engagement_score', 'dropout_risk'
    ]]
    .mean()
    .rename(columns={'dropout_risk': 'dropout_rate'})
    .sort_values('dropout_rate', ascending=False)
    .round(3)
)
program_profile['dropout_rate'] = (program_profile['dropout_rate'] * 100).round(1)

print('=== Profil complet par programme ===')
print(program_profile.to_string())

program_profile.to_csv('data/processed/tp2_program_profiles.csv')
df.groupby('semester')['dropout_risk'].mean().mul(100).round(2).to_csv(
    'data/processed/tp2_semester_dropout_rates.csv'
)
print('\nFichiers EDA sauvegardes.')

## 6. Synthese EDA

| Dimension | Observation | Signification metier |
|---|---|---|
| **Desequilibre** | ~8.5 % de positifs | Necessite recall > accuracy comme metrique cle |
| **Correlation cible** | `engagement_score`, `prior_average`, `continuous_assessment` les plus correles | Ce sont les variables a surveiller en priorite |
| **Biais social** | `parental_education=secondary` double presque le risque | Risque d'amplification des inegalites — mettre en place un audit equite |
| **Tests statistiques** | Toutes les variables cles montrent des differences significatives (p < 0.001) | Le signal statistique est fort et exploitable |
| **Profil a risque** | Moins d'assiduite, plus de retards, moins de notes, plus de stress | Portrait coherent, signal actionnable par les equipes |